# Log-sum-exp pooling: 29 layers to 15 windows, then 4 × 128 MLP, focal loss

Identical to the log-sum-exp 4 × 128 run (same pooling, six-source split, 40 epochs, checkpoint selection, held-out TriviaQA evaluation), except the BCE loss is replaced by standard binary focal loss (Lin et al., 2017) with α = 0.25 and γ = 2.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 153de1f
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_logsumexp_4x128_focal')
command = [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0', '--hidden-dim', '128', '--hidden-layers', '4', '--pooling', 'logsumexp1d', '--pool-kernel-size', '3', '--pool-stride', '2', '--loss', 'focal', '--focal-alpha', '0.25', '--focal-gamma', '2.0']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Pooled layers:', summary['pooled_layers'], 'channels:', summary['pooling_output_channels'], 'MLP input:', summary['mlp_input_dimension'])
print('Loss:', summary['classifier_loss'], 'alpha:', summary['focal_alpha'], 'gamma:', summary['focal_gamma'])
print(json.dumps(summary['target_metrics'], indent=2))
display(results.head())